# 워크플로우와 에이전트 경계

누가 다음 단계를 정하는지 사건별로 표시해 두 구조의 경계를 확인합니다.

## 실행 환경 준비

외부 API를 호출하지 않습니다. 교육용 고정 사례를 읽고 분류하므로 실행할 때마다 같은 결과가 나옵니다.

## 코드를 읽기 전에

첫 표의 한 행은 하나의 시스템 사례입니다. `decision_owner`는 다음 행동을 고른 주체, `unknown_steps`는 시작 전에 필요한 단계 수를 알 수 있는지, `tool_count`는 사용 가능한 도구 수를 나타냅니다. 이 세 값 가운데 분류 함수가 실제로 읽는 것은 앞의 두 값입니다. 따라서 도구 수를 바꾼 뒤 분류가 그대로라면, 도구의 존재와 결정권이 서로 다른 개념임을 코드로 확인한 것입니다.

`architecture(case)`는 사례 사전을 입력받아 문자열 하나를 반환합니다. 모델이 다음 행동을 정하고 단계 수도 미리 알 수 없을 때만 에이전트 후보로 표시합니다. 이 반환값은 제품 이름을 판정한 사실이 아니라 수업에서 정한 경계 규칙을 고정 사례에 적용한 결과입니다.

In [1]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
from collections import Counter
from collections.abc import Mapping
from pathlib import Path
import sys
from typing import TypedDict

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part5/ch17-agent-overview").is_dir())
LAB = ROOT / "labs/part5/ch17-agent-overview"
print("Python:", sys.version.split()[0], "| 실습 폴더:", LAB.relative_to(ROOT))

Python: 3.12.13 | 실습 폴더: labs/part5/ch17-agent-overview


## 사례 준비

`decision_owner`는 다음 행동을 정한 주체입니다. `code`면 미리 작성한 경로, `model`이면 입력과 관찰 결과를 본 모델의 선택입니다.

In [2]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
class Case(TypedDict):
    name: str
    steps: int | None
    decision_owner: str
    unknown_steps: bool

cases: list[Case] = [
    {"name": "정해진 환불 절차", "steps": 4, "decision_owner": "code", "unknown_steps": False},
    {"name": "여러 파일의 결함 수정", "steps": None, "decision_owner": "model", "unknown_steps": True},
    {"name": "문서 요약 뒤 형식 검사", "steps": 3, "decision_owner": "code", "unknown_steps": False},
    {"name": "낯선 장애 원인 조사", "steps": None, "decision_owner": "model", "unknown_steps": True},
]
print(*cases, sep="\n")

{'name': '정해진 환불 절차', 'steps': 4, 'decision_owner': 'code', 'unknown_steps': False}
{'name': '여러 파일의 결함 수정', 'steps': None, 'decision_owner': 'model', 'unknown_steps': True}
{'name': '문서 요약 뒤 형식 검사', 'steps': 3, 'decision_owner': 'code', 'unknown_steps': False}
{'name': '낯선 장애 원인 조사', 'steps': None, 'decision_owner': 'model', 'unknown_steps': True}


## 경계 분류

도구 사용 여부보다 다음 단계의 결정 주체를 먼저 봅니다.

In [3]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
def architecture(case: Mapping[str, object]) -> str:
    if case["decision_owner"] == "model" and case["unknown_steps"] is True:
        return "에이전트 후보"
    return "워크플로우"

for case in cases:
    print(case["name"], "→", architecture(case))

정해진 환불 절차 → 워크플로우
여러 파일의 결함 수정 → 에이전트 후보
문서 요약 뒤 형식 검사 → 워크플로우
낯선 장애 원인 조사 → 에이전트 후보


## 값 하나를 바꿔 비교

같은 환불 절차에서 도구를 세 개로 늘려도 분기를 코드가 정하면 분류는 바뀌지 않습니다.

In [4]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
refund = cases[0] | {"tool_count": 3}
print("도구 수:", refund["tool_count"])
print("결정 주체:", refund["decision_owner"])
print("분류:", architecture(refund))
assert architecture(refund) == "워크플로우"

도구 수: 3
결정 주체: code
분류: 워크플로우


## 전체 사례 집계

분류 결과는 교육용 사례의 필드에서 계산한 값입니다. 실제 제품의 내부 구조를 측정한 결과가 아닙니다.

In [5]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
counts = Counter(architecture(case) for case in cases)
print(dict(counts))
print("판정 기준: 다음 단계 결정 주체 + 필요한 단계의 사전 예측 가능성")
assert counts == {"워크플로우": 2, "에이전트 후보": 2}

{'워크플로우': 2, '에이전트 후보': 2}
판정 기준: 다음 단계 결정 주체 + 필요한 단계의 사전 예측 가능성


## 다른 상황에 적용하기

고객 문의 시스템이 모델로 문의 유형을 분류하더라도, 코드가 준비된 세 경로 가운데 하나만 실행한다면 제한된 라우팅 워크플로우로 설명할 수 있습니다. 반대로 모델이 검색 결과를 보고 새로운 조사 항목을 만들고 도구를 반복 선택한다면 에이전트 쪽에 가까워집니다. 중간 형태에서는 ‘모델이 무엇을 고르는가’와 ‘코드가 무엇을 제한하는가’를 따로 기록해야 합니다.